# 05 model evaluation

**Owner:** Denis. Original notebook cells: 48, 49.

Run in the shared Databricks session using `run_pipeline.ipynb` or the documented stage runner. This is not a standalone clean-kernel notebook. Saved outputs have been removed for public-repository safety. See `docs/execution_order.md`.

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

# Use the untouched test partition
X_test_raw = test[features]
y_test_final = test["target_no_show"].astype(int).to_numpy()

# Logistic Regression predictions
baseline_test_probs = baseline_model.predict_proba(
    X_test_raw
)[:, 1]

baseline_threshold = float(
    baseline_metadata["classification_threshold"]
)

baseline_test_preds = (
    baseline_test_probs >= baseline_threshold
).astype(int)

# MLP predictions using the verified MLflow model
X_test_nn = prepare_features(test)

mlp_test_probs = loaded_mlp.predict(
    X_test_nn,
    batch_size=256,
    verbose=0
).ravel()

mlp_test_preds = (
    mlp_test_probs >= final_mlp_threshold
).astype(int)

# Calculate identical metrics for both models
def evaluate_model(name, y_true, predictions, probabilities):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_true, predictions, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_true, probabilities),
        "PR-AUC": average_precision_score(
            y_true, probabilities
        )
    }

test_results = pd.DataFrame([
    evaluate_model(
        "Logistic Regression",
        y_test_final,
        baseline_test_preds,
        baseline_test_probs
    ),
    evaluate_model(
        f"MLP ({best_model_name})",
        y_test_final,
        mlp_test_preds,
        mlp_test_probs
    )
])

print("=== FINAL HELD-OUT TEST RESULTS ===")
print("Test appointments:", len(y_test_final))
print("Actual no-show rate:", round(y_test_final.mean() * 100, 2), "%")

display(test_results.round(4))

# Confusion matrices for both models
for name, predictions in [
    ("Logistic Regression", baseline_test_preds),
    (f"MLP ({best_model_name})", mlp_test_preds)
]:
    print(f"\n=== {name} ===")

    print(classification_report(
        y_test_final,
        predictions,
        target_names=["Attended", "No-show"],
        zero_division=0
    ))

    cm = confusion_matrix(y_test_final, predictions)

    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Attended", "No-show"]
    ).plot(cmap="Blues", values_format="d")

    plt.title(f"{name} - Held-Out Test Confusion Matrix")
    plt.tight_layout()
    plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Create an error-analysis table without patient identifiers
error_df = test[
    ["Age", "LeadTime_Days", "target_no_show"]
].copy().reset_index(drop=True)

error_df["Prediction"] = mlp_test_preds
error_df["No_Show_Probability"] = mlp_test_probs

# Classify the prediction outcomes
conditions = [
    (error_df["target_no_show"] == 0) & (error_df["Prediction"] == 0),
    (error_df["target_no_show"] == 0) & (error_df["Prediction"] == 1),
    (error_df["target_no_show"] == 1) & (error_df["Prediction"] == 0),
    (error_df["target_no_show"] == 1) & (error_df["Prediction"] == 1)
]

labels = [
    "True Negative",
    "False Positive",
    "False Negative",
    "True Positive"
]

error_df["Outcome"] = np.select(
    conditions, labels, default="Unknown"
)

print("=== PREDICTION OUTCOMES ===")
print(error_df["Outcome"].value_counts())

# Create age groups
error_df["Age_Group"] = pd.cut(
    error_df["Age"],
    bins=[-1, 17, 29, 44, 59, 74, np.inf],
    labels=["0-17", "18-29", "30-44", "45-59", "60-74", "75+"]
)

# Create booking lead-time groups
error_df["LeadTime_Group"] = pd.cut(
    error_df["LeadTime_Days"],
    bins=[-1, 0, 2, 7, 14, 30, 60, np.inf],
    labels=[
        "Same day", "1-2 days", "3-7 days",
        "8-14 days", "15-30 days",
        "31-60 days", "61+ days"
    ]
)

def analyze_group(column):
    rows = []

    for group, subset in error_df.groupby(
        column, observed=True
    ):
        tn, fp, fn, tp = confusion_matrix(
            subset["target_no_show"],
            subset["Prediction"],
            labels=[0, 1]
        ).ravel()

        rows.append({
            "Group": str(group),
            "Appointments": len(subset),
            "False Positives": int(fp),
            "False Negatives": int(fn),
            "Precision": tp / (tp + fp) if tp + fp else 0,
            "Recall": tp / (tp + fn) if tp + fn else 0,
            "False Positive Rate": fp / (fp + tn) if fp + tn else 0
        })

    return pd.DataFrame(rows)

age_errors = analyze_group("Age_Group")
lead_errors = analyze_group("LeadTime_Group")

print("\n=== ERROR ANALYSIS BY AGE ===")
display(age_errors.round(4))

print("\n=== ERROR ANALYSIS BY BOOKING LEAD TIME ===")
display(lead_errors.round(4))

# Visualize recall by age group
plt.figure(figsize=(9, 4))
plt.bar(age_errors["Group"], age_errors["Recall"])
plt.title("MLP No-Show Recall by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Recall")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()